<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Deep Learning & Neural Networks** · Week 1 · Neural Networks from Scratch · Lesson 3 of 24

# Loss Functions and Gradient Descent

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks).

**Learning objective:** Measure how wrong a network is with mean squared error and cross-entropy, and train a single neuron with gradient descent.

## A score for being wrong

To improve, a network needs one number that says how wrong it is: the **loss**. Training means changing the weights to make the loss smaller.

- **Mean squared error (MSE)** — for numbers: the average of (prediction − truth)². You used it in Math & Python for AI.
- **Cross-entropy** — for classes: punishes confident wrong answers very hard.

> binary cross-entropy = −[ y × log(p) + (1 − y) × log(1 − p) ]

where y is the true label (0 or 1) and p is the predicted probability of class 1.

In [ ]:
import numpy as np

def bce(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

for p in [0.99, 0.9, 0.6, 0.4, 0.1, 0.01]:
    print(f"true label 1, predicted {p:4.2f} -> loss {bce(1, p):.3f}")

A confident correct answer (0.99) costs almost nothing (about 0.01). A confident wrong answer (0.01) costs about **4.6**. Cross-entropy teaches the network not to be confidently wrong. For several classes, **categorical cross-entropy** is −log(probability given to the true class).

## Gradient descent, again

The loss depends on every weight. The **gradient** tells us, for each weight, which direction makes the loss go **up** — so we step the other way:

> new weight = old weight − learning rate × gradient

This is exactly the gradient descent you did by hand for linear regression. Deep learning uses the same idea for millions of weights at once.

## Train one neuron (logistic regression)

A single neuron with a sigmoid is logistic regression. For cross-entropy loss, the gradient has a beautifully simple form: (prediction − truth) × input.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, X_test.shape)
plt.scatter(X[:, 0], X[:, 1], c=y, s=8)
plt.title("Two moons: a pattern a straight line cannot separate")
plt.show()

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros(2)
b = 0.0
lr = 0.5
history = []
for epoch in range(500):
    p = sigmoid(X_train @ w + b)
    loss = bce(y_train, np.clip(p, 1e-9, 1 - 1e-9)).mean()
    history.append(loss)
    grad_w = X_train.T @ (p - y_train) / len(y_train)
    grad_b = (p - y_train).mean()
    w -= lr * grad_w
    b -= lr * grad_b

acc = ((sigmoid(X_test @ w + b) > 0.5) == y_test).mean()
print("final loss:", round(history[-1], 3), " test accuracy:", acc)
plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.show()

The loss falls quickly then levels off, and the single neuron reaches about **0.86** test accuracy. It cannot do better: one neuron draws one **straight** line, and the moons are curved. Next lesson, a hidden layer fixes that.

## Check a gradient by nudging

You can always check a gradient numerically: nudge a weight a tiny bit and see how the loss changes.

In [ ]:
def mean_loss(w, b):
    p = np.clip(sigmoid(X_train @ w + b), 1e-9, 1 - 1e-9)
    return bce(y_train, p).mean()

eps = 1e-5
w_plus = w.copy(); w_plus[0] += eps
w_minus = w.copy(); w_minus[0] -= eps
numeric = (mean_loss(w_plus, b) - mean_loss(w_minus, b)) / (2 * eps)
analytic = X_train[:, 0] @ (sigmoid(X_train @ w + b) - y_train) / len(y_train)
print(round(numeric, 6), round(analytic, 6))

The two numbers agree — our formula is right. This **gradient check** is a classic debugging trick.

## Mini-batches and epochs

Real datasets are too big to use all at once. **Stochastic / mini-batch gradient descent** uses a small random batch (for example 32 or 64 examples) for each step. One pass through all the batches is an **epoch**. Batches make training faster and the noise even helps escape poor solutions.

## Key terms

- **Loss** — one number measuring how wrong the predictions are.
- **Cross-entropy** — the standard loss for classification.
- **Gradient** — the direction in which each weight increases the loss.
- **Learning rate** — the size of each step.
- **Mini-batch** — a small group of examples used for one update.
- **Epoch** — one full pass through the training data.

**Takeaway:** Training is a loop — predict, measure the loss, compute gradients, step downhill — and cross-entropy is the loss for classification.

## Try it yourself

1. Try learning rates 0.05, 0.5 and 5. Plot the three loss curves on one chart.
2. Change the training loop to use random mini-batches of 32 rows. Does it still learn?
3. Compute the categorical cross-entropy when the true class gets probability 0.7, then 0.05.
4. Plot the straight decision line w₁x₁ + w₂x₂ + b = 0 on top of the moons scatter plot.

---
Finished? Go back to [Deep Learning & Neural Networks on WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*